In [2]:
import torch

In [3]:
def get_normalize(features: torch.Tensor):
    # 1. Считаем среднее по каналам
    mean = features.mean(dim=(0, 2, 3))
    
    # 2. Считаем стандартное отклонение по каналам (несмещенное по умолчанию в PyTorch)
    std = features.std(dim=(0, 2, 3), correction=1)
    
    return mean, std

In [6]:
t = torch.tensor([[[[1.,2.],[3.,4.]],[[10.,20.],[30.,40.]]]])
get_normalize(t)

(tensor([ 2.5000, 25.0000]), tensor([ 1.2910, 12.9099]))

In [ ]:
import torchvision.transforms as T

def get_augmentations(train: bool = True) -> T.Compose:
    # Константы нормализации для CIFAR10, указанные на подсказке ко 2-му скриншоту
    means = (0.49139968, 0.48215841, 0.44653091)
    stds = (0.24703223, 0.24348513, 0.26158784)
    
    if train:
        # Аугментации для обучающей выборки (с добавлением случайного отражения)
        return T.Compose([
            T.Resize((224, 224)),
            T.RandomHorizontalFlip(), # Любая базовая аугментация с занятия
            T.ToTensor(),
            T.Normalize(mean=means, std=stds)
        ])
    else:
        # Аугментации для тестовой/валидационной выборки (без случайных изменений)
        return T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=means, std=stds)
        ])

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> torch.Tensor:
    # Переводим модель в режим оценки (выключает Dropout и BatchNorm)
    model.eval()
    model.to(device)
    
    # 1. Создаем пустой список для хранения предсказаний
    all_predictions = []
    
    # Отключаем подсчет градиентов для ускорения инференса
    with torch.no_grad():
        # 2. Проитерируемся по даталоадеру
        for x, _ in loader:
            # Переносим входной батч картинок на целевое устройство (процессор или видеокарту)
            x = x.to(device)
            
            # 3. Делаем forward pass для батча
            logits = model(x)
            
            # Считаем классы как аргмакс по выходу (вдоль размерности классов dim=1)
            y_pred = torch.argmax(logits, dim=1)
            
            # Добавляем тензор с предсказаниями в список
            all_predictions.append(y_pred)
            
    # 4. Делаем конкатенацию всех предсказаний в один длинный тензор
    # Переносим итоговый тензор на CPU (обычно хорошая практика для чекеров, 
    # но если чекер требует оставить на девайсе, можно .cpu() убрать)
    final_predictions = torch.cat(all_predictions, dim=0)
    
    return final_predictions

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

In [9]:
def create_simple_conv_cifar() -> nn.Module:
    """
    Создает модель с ровно двумя сверточными слоями для CIFAR10.
    """
    model = nn.Sequential(
        # Блок свертки 1
        nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1), # Вход: 3x32x32
        nn.BatchNorm2d(32),   # Стабилизирует обучение
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2), # Выход: 32x16x16
        nn.Dropout2d(0.1),

        # Блок свертки 2 (последний сверточный по ограничению)
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2), # Выход: 64x8x8
        nn.Dropout2d(0.2),

        # Классификатор (Полносвязные слои)
        nn.Flatten(), # Выпрямляем: 64 * 8 * 8 = 4096
        nn.Linear(64 * 8 * 8, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 10) # 10 классов CIFAR10
    )
    return model

In [10]:
# --- 2. Функция predict из предыдущего задания ---

def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> torch.Tensor:
    model.eval()
    model.to(device)
    all_predictions = []
    
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            logits = model(x)
            y_pred = torch.argmax(logits, dim=1)
            all_predictions.append(y_pred)
            
    return torch.cat(all_predictions, dim=0)

In [3]:
# --- 3. Вспомогательные функции для тренировочного цикла ---

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    return total_loss / len(loader), correct / total

def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = criterion(output, y)
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    return total_loss / len(loader), correct / total

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

# Загрузка данных с базовой нормализацией для CIFAR10
# Аугментации на тренировочной выборке помогут быстрее набрать честные 70%+ без переобучения
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

model = create_simple_conv_cifar().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 5-7 эпох с AdamW обычно достаточно для достижения 72-75% точности
epochs = 7 
for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    te_loss, te_acc = evaluate_epoch(model, test_loader, criterion, device)
    print(f"Эпоха {epoch} | Train Acc: {tr_acc*100:.2f}% | Test Acc: {te_acc*100:.2f}%")

    # Если преодолели порог, делаем предсказания и сохраняем по правилам
    if te_acc >= 0.70:
        print("Целевая метрика в 70% достигнута!")
        
        # Получаем итоговый тензор предсказаний длины N (10000 для тест-сета CIFAR10)
        predictions = predict(model, test_loader, device)
        
        # Сохраняем строго через torch.save на диск в текущую папку
        # Переносим на CPU для гарантированной совместимости с чекером
        torch.save(predictions.cpu(), 'predictions.pt')
        print("Тензор предсказаний успешно записан в файл 'predictions.pt'")
        break

Используем устройство: cpu


100%|██████████| 170M/170M [01:52<00:00, 1.52MB/s] 


Эпоха 1 | Train Acc: 42.82% | Test Acc: 59.12%
Эпоха 2 | Train Acc: 53.31% | Test Acc: 65.19%
Эпоха 3 | Train Acc: 57.15% | Test Acc: 67.47%
Эпоха 4 | Train Acc: 59.45% | Test Acc: 68.66%
Эпоха 5 | Train Acc: 61.25% | Test Acc: 70.37%
Целевая метрика в 70% достигнута!
Тензор предсказаний успешно записан в файл 'predictions.pt'


In [1]:
@torch.no_grad()
def predict_tta(model: nn.Module, loader: DataLoader, device: torch.device, iterations: int = 2) -> torch.Tensor:
    # Переводим модель в режим инференса
    model.eval()
    model.to(device)
    
    # Список, где будем хранить итоговые тензоры логитов размером [N, C] для каждой итерации
    iteration_logits = []
    
    # 1. Запускаем цикл по количеству итераций
    for _ in range(iterations):
        single_prediction = []
        
        # 2. Внутри цикла проитерируемся по даталоадеру
        for x, _ in loader:
            x = x.to(device)
            
            # Получаем сырые выходы нейросети (логиты)
            logits = model(x)
            single_prediction.append(logits)
            
        # 3. Записываем ответы в один большой тензор размера [N, C] для текущей итерации
        # Используем конструкцию по подсказке ко второму скриншоту
        full_iteration_logits = torch.vstack(single_prediction)
        iteration_logits.append(full_iteration_logits)
        
    # 4. Собираем тензоры всех итераций в один огромный тензор
    # Сначала упаковываем в [iterations, N, C], затем меняем оси на [N, C, iterations]
    stacked_logits = torch.stack(iteration_logits, dim=0) # Размерность: [iterations, N, C]
    permuted_logits = stacked_logits.permute(1, 2, 0)     # Размерность: [N, C, iterations]
    
    # Усредняем его по оси итераций (dim=2), получая тензор размера [N, C]
    mean_logits = permuted_logits.mean(dim=2)
    
    # 5. Предсказываем классы как аргмакс по выходам вдоль оси классов (dim=1)
    final_classes = torch.argmax(mean_logits, dim=1)
    
    return final_classes

NameError: name 'torch' is not defined

In [ ]:
def create_advanced_conv_cifar() -> nn.Module:
    model = nn.Sequential(
        # Блок 1: 32x32 -> 16x16
        nn.Conv2d(3, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout2d(0.2),

        # Блок 2: 16x16 -> 8x8
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout2d(0.3),

        # Блок 3: 8x8 -> 4x4
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout2d(0.4),

        # Полносвязный классификатор
        nn.Flatten(), # 256 * 4 * 4 = 4096
        nn.Linear(4096, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 10)
    )
    return model